# Chunk Quality Tuning Advisor

This notebook assesses chunk quality using the same heuristic logic used in ingestion and provides tuning advice aligned with `docs/CHUNK_VALIDATION_HEURISTIC.md`.

It can analyse chunk text from either:
- ChromaDB chunk collection (recommended), or
- `logs/ingest_audit.jsonl` heuristic events (if available).

In [ ]:
from __future__ import annotations

import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd

import chromadb

from scripts.ingest.vectors import compute_chunk_quality_heuristic
from scripts.utils.config import BaseConfig

# Guide-aligned defaults
THRESHOLDS = {
    'min_length': 100,
    'max_length': 2000,
    'min_stopword_ratio': 0.15,
    'min_entropy': 3.0,
    'boilerplate_threshold': 3,
}

MAX_ROWS = 5000

config = BaseConfig()
rag_data_path = Path(config.rag_data_path)
chroma_path = rag_data_path / 'chromadb'
audit_path = Path.cwd().parent / 'logs' / 'ingest_audit.jsonl' if Path.cwd().name == 'notebooks' else Path.cwd() / 'logs' / 'ingest_audit.jsonl'

print(f'Chroma path: {chroma_path}')
print(f'Chunk collection: {config.chunk_collection_name}')
print(f'Audit path: {audit_path}')
print(f'Thresholds: {THRESHOLDS}')

In [ ]:
STOPWORDS = {
    'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'i', 'it', 'for', 'not', 'on', 'with', 'he',
    'as', 'you', 'do', 'at', 'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she', 'or',
    'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their', 'what', 'so', 'up', 'out', 'if', 'about',
    'who', 'get', 'which', 'go', 'me', 'when', 'make', 'can', 'like', 'time', 'no', 'just', 'him', 'know',
    'take', 'people', 'into', 'year', 'your', 'good', 'some', 'could', 'them', 'see', 'other', 'than',
    'then', 'now', 'look', 'only', 'come', 'its', 'over', 'think', 'also', 'back', 'after', 'use', 'two',
    'how', 'our', 'work', 'first', 'well', 'way', 'even', 'new', 'want', 'because', 'any', 'these',
    'give', 'day', 'most', 'us'
}

BOILERPLATE_PATTERNS = [
    'copyright', 'all rights reserved', 'terms of service', 'privacy policy', 'cookie settings',
    'contact us', '©', 'trademark', 'click here', 'next page', 'previous', 'home page', 'navigation',
    'breadcrumb', 'sitemap', 'skip to', 'login', 'sign up', 'subscribe'
]

def compute_metrics(text: str) -> dict[str, Any]:
    length = len(text)
    words = text.lower().split()
    stopword_ratio = 0.0
    if words:
        stopword_ratio = sum(1 for word in words if word in STOPWORDS) / len(words)

    text_lower = text.lower()
    boilerplate_count = sum(1 for pattern in BOILERPLATE_PATTERNS if pattern in text_lower)

    entropy = 0.0
    if length > 0:
        char_counts = Counter(text_lower)
        entropy = -sum((count / length) * math.log2(count / length) for count in char_counts.values())

    skip, confidence, reason = compute_chunk_quality_heuristic(text)

    return {
        'length': length,
        'word_count': len(words),
        'stopword_ratio': stopword_ratio,
        'boilerplate_count': boilerplate_count,
        'entropy': entropy,
        'skip_validation': bool(skip),
        'confidence': float(confidence),
        'reason': reason,
    }

In [ ]:
def load_chunks_from_chromadb(limit: int = MAX_ROWS) -> pd.DataFrame:
    if not chroma_path.exists():
        print('Chroma path does not exist; skipping Chroma load.')
        return pd.DataFrame()

    client = chromadb.PersistentClient(path=str(chroma_path))
    collection = client.get_collection(config.chunk_collection_name)

    result = collection.get(limit=limit, include=['documents', 'metadatas'])
    ids = result.get('ids') or []
    docs = result.get('documents') or []
    metas = result.get('metadatas') or []

    rows: list[dict[str, Any]] = []
    for idx, chunk_id in enumerate(ids):
        text = docs[idx] if idx < len(docs) and docs[idx] else ''
        if not text:
            continue

        meta = metas[idx] if idx < len(metas) and metas[idx] else {}
        metrics = compute_metrics(text)

        rows.append({
            'chunk_id': chunk_id,
            'doc_id': meta.get('doc_id', ''),
            'doc_type': meta.get('doc_type', ''),
            'chunk_type': meta.get('chunk_type', 'regular'),
            **metrics,
        })

    return pd.DataFrame(rows)

df = load_chunks_from_chromadb(limit=MAX_ROWS)
print(f'Loaded {len(df)} chunks from ChromaDB')
df.head(3)

In [ ]:
def load_heuristic_events_from_audit(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    rows: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                continue

            if event.get('event') != 'chunk_heuristic_skip':
                continue

            data = event.get('data', {})
            rows.append({
                'chunk_id': data.get('chunk_id', ''),
                'doc_id': data.get('doc_id', ''),
                'skip_validation': True,
                'confidence': data.get('confidence'),
                'reason': data.get('reason', 'heuristic_pass'),
            })

    return pd.DataFrame(rows)

audit_df = load_heuristic_events_from_audit(audit_path)
print(f'Loaded {len(audit_df)} heuristic skip events from audit log')
audit_df.head(3)

In [ ]:
if df.empty:
    print('No chunk records loaded from ChromaDB. Run ingestion first, then rerun this notebook.')
else:
    skip_rate = (df['skip_validation'].mean() * 100)
    avg_conf = df['confidence'].mean()

    summary = pd.DataFrame({
        'metric': [
            'chunks_analysed',
            'skip_rate_pct',
            'validate_rate_pct',
            'avg_confidence',
            'avg_length_chars',
            'median_length_chars',
            'avg_stopword_ratio',
            'avg_entropy',
        ],
        'value': [
            len(df),
            round(skip_rate, 2),
            round(100 - skip_rate, 2),
            round(avg_conf, 3),
            round(df['length'].mean(), 1),
            round(df['length'].median(), 1),
            round(df['stopword_ratio'].mean(), 3),
            round(df['entropy'].mean(), 3),
        ]
    })

    display(summary)

    print('Top validation reasons:')
    display(df['reason'].value_counts().head(10).rename_axis('reason').reset_index(name='count'))

In [ ]:
if df.empty:
    print('No tuning advice available without chunk data.')
else:
    reasons = df['reason'].value_counts(normalize=True)

    def pct(prefix: str) -> float:
        return float(reasons[[idx.startswith(prefix) for idx in reasons.index]].sum() * 100)

    too_short_pct = pct('too_short')
    too_long_pct = pct('too_long')
    low_stopword_pct = pct('low_stopword_ratio_')
    low_entropy_pct = pct('low_entropy_')
    boilerplate_pct = pct('boilerplate_detected_')

    skip_rate = float(df['skip_validation'].mean() * 100)

    print('Tuning advice (guide-aligned)')
    print('=' * 72)
    print(f'Current skip rate: {skip_rate:.1f}%')
    print()

    if skip_rate < 30:
        print('- Skip rate is low. Consider less conservative tuning for mixed technical corpora:')
        print('  - MIN_STOPWORD_RATIO: 0.15 -> 0.10')
        print('  - MIN_ENTROPY: 3.0 -> 2.5')
        print('  - BOILERPLATE_THRESHOLD: 3 -> 4')

    if too_short_pct > 15:
        print(f'- {too_short_pct:.1f}% flagged as too_short:')
        print('  - Increase chunk size or reduce aggressive separators in chunking.')
        print('  - Review extraction quality for truncated/fragmented source text.')

    if too_long_pct > 10:
        print(f'- {too_long_pct:.1f}% flagged as too_long:')
        print('  - Decrease chunk size or increase splitting granularity.')

    if low_stopword_pct > 20:
        print(f'- {low_stopword_pct:.1f}% flagged with low stopword ratio:')
        print('  - For technical content, lowering MIN_STOPWORD_RATIO may be appropriate.')
        print('  - Verify tokenisation quality and language consistency.')

    if low_entropy_pct > 15:
        print(f'- {low_entropy_pct:.1f}% flagged with low entropy:')
        print('  - Check for repetitive noise, OCR artefacts, or template-heavy exports.')
        print('  - Consider pre-cleaning before chunking.')

    if boilerplate_pct > 10:
        print(f'- {boilerplate_pct:.1f}% flagged as boilerplate:')
        print('  - Expand boilerplate patterns for your domain in vectors.py.')
        print('  - Improve HTML/PDF cleaning to remove nav/copyright text earlier.')

    print()
    print('Suggested validation profile:')
    if skip_rate < 30:
        print('- Use the aggressive profile from the guide to increase skip rate.')
    elif skip_rate > 80:
        print('- Use a more conservative profile to reduce false skips.')
        print('  - MIN_STOPWORD_RATIO: 0.15 -> 0.20')
        print('  - MIN_ENTROPY: 3.0 -> 3.5')
        print('  - BOILERPLATE_THRESHOLD: 3 -> 2')
    else:
        print('- Current tuning is balanced for many corpora; apply targeted threshold changes only.')

In [ ]:
if not df.empty:
    by_chunk_type = (
        df.groupby('chunk_type')
        .agg(
            chunks=('chunk_id', 'count'),
            skip_rate_pct=('skip_validation', lambda s: round(float(s.mean() * 100), 2)),
            avg_confidence=('confidence', lambda s: round(float(s.mean()), 3)),
            avg_length=('length', lambda s: round(float(s.mean()), 1)),
        )
        .reset_index()
        .sort_values('chunks', ascending=False)
    )
    print('Chunk-type profile (useful when parent/child chunking is enabled):')
    display(by_chunk_type)

    print('Next step: tune thresholds, re-run ingestion on a sample corpus, then compare this table again.')